# Experiment 5: Model Selection
In this final experiment of the series, we compare various machine learning algorithms to determine which one performs best for YouTube comment sentiment analysis.

### Models to compare:
1. **Logistic Regression (LoR)**: Linear model, fast and often a strong baseline for text.
2. **Naive Bayes (MultinomialNB)**: Probabilistic model, extremely fast and effective for text.
3. **Random Forest (RF)**: Ensemble of decision trees.
4. **XGBoost & LightGBM**: Gradient Boosted Decision Trees, state-of-the-art for many tabular/structured tasks.
5. **Support Vector Machine (SVM)**: Effective in high-dimensional spaces (like text).
6. **K-Nearest Neighbors (KNN)**: Simple distance-based model.

### Why compare models?
Different algorithms have different biases. Some handle high-dimensional sparse data (like text) better, while others are better at capturing non-linear relationships.

In [1]:
import pandas as pd
import numpy as np
import mlflow
import os
import tempfile
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                              recall_score, classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_sample_weight
from scipy.sparse import hstack
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
try:
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier
except ImportError:
    print("XGBoost or LightGBM not installed. Skipping those.")

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Model Selection")


c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/8', creation_time=1776071928841, experiment_id='8', last_update_time=1776071928841, lifecycle_stage='active', name='Model Selection', tags={}, workspace='default'>

In [2]:
# Load and preprocess
data = pd.read_csv(r'../data/processed/dataset.csv')
data = data.dropna(subset=['text_processed', 'sentiment']).drop_duplicates()

X_text = data['text_processed']
X_numeric = data[['char_count', 'word_count', 'avg_word_len']].values
y = data['sentiment']

# Label encoding for y if needed by some models (like XGBoost)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_numeric, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Text + Scaling setup
tfidf = TfidfVectorizer(max_features=500, ngram_range=(1,1))
scaler = StandardScaler()

X_text_train_vec = tfidf.fit_transform(X_text_train)
X_text_test_vec = tfidf.transform(X_text_test)
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)

X_train_final = hstack([X_text_train_vec, X_num_train_scaled])
X_test_final = hstack([X_text_test_vec, X_num_test_scaled])

# Special data for Naive Bayes (No negative values)
nb_scaler = MinMaxScaler()
X_num_train_nb = nb_scaler.fit_transform(X_num_train)
X_num_test_nb = nb_scaler.transform(X_num_test)
X_train_nb = hstack([X_text_train_vec, X_num_train_nb])
X_test_nb = hstack([X_text_test_vec, X_num_test_nb])

In [3]:
# Define the core models we want to evaluate
base_models = [
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=42)),
    ("NaiveBayes", MultinomialNB(alpha=1.0)),
    ("ComplementNB", ComplementNB()),
    ("RandomForest", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("SVM", SVC(kernel='rbf', random_state=42, probability=True)),
    ("LinearSVC", LinearSVC(random_state=42, max_iter=2000)),
    ("KNN_k5", KNeighborsClassifier(n_neighbors=5)),
]

# Add Gradient Boosting
try:
    base_models.append(("LightGBM", LGBMClassifier(random_state=42, verbosity=-1)))
    base_models.append(("XGBoost", XGBClassifier(random_state=42, eval_metric='mlogloss')))
except:
    pass

# Define the three distinct scenarios requested:
# 1. Baseline (No resampling, No weights)
# 2. Class Weights (Only for supported models, No resampling)
# 3. Resampling (SMOTE variants, No weights)
strategies = ["Baseline", "ClassWeights", "SMOTE", "SMOTETomek", "BorderlineSMOTE", "ADASYN"]

for model_name, base_clf in base_models:
    for strategy in strategies:
        
        # ── Logic Filtering ──
        has_weight_support = hasattr(base_clf, "class_weight") or model_name == "XGBoost"
        
        # 1. Skip ClassWeights for models that don't support it (KNN/NB)
        if strategy == "ClassWeights" and not has_weight_support:
            continue
            
        # 2. Keep BorderlineSMOTE and ADASYN specialized for SVM (as per previous industry request)
        # or apply them generally if desired. Here we keep them for SVM to avoid massive run times.
        if strategy in ["BorderlineSMOTE", "ADASYN"] and model_name != "SVM":
            continue
            
        run_name = f"{model_name}_{strategy}"
        
        with mlflow.start_run(run_name=run_name):
            # Setup Training Data (NB variants use MinMaxScaler)
            is_nb = "NB" in model_name or "Naive" in model_name
            X_tr = X_train_nb if is_nb else X_train_final
            X_ts = X_test_nb if is_nb else X_test_final
            
            # Configure the classifier instance
            # We must use a fresh clone/instance to avoid parameter bleed between runs
            from sklearn.base import clone
            curr_clf = clone(base_clf)
            
            # Scenario A: Class Weights Only
            if strategy == "ClassWeights":
                if model_name != "XGBoost":
                    curr_clf.set_params(class_weight='balanced')
            
            # Scenario B: Resampling Only (Ensure weights are None/Default)
            resampler = None
            if strategy == "SMOTE":
                resampler = SMOTE(random_state=42)
            elif strategy == "SMOTETomek":
                resampler = SMOTETomek(random_state=42)
            elif strategy == "BorderlineSMOTE":
                resampler = BorderlineSMOTE(random_state=42)
            elif strategy == "ADASYN":
                resampler = ADASYN(random_state=42)

            if resampler:
                # Industry standard: wrap in pipeline to prevent leakage
                model = Pipeline([('resample', resampler), ('clf', curr_clf)])
            else:
                model = curr_clf

            # ── Train ─────────────────────────────────────────────────────────────────
            if model_name == "XGBoost" and strategy == "ClassWeights":
                # Apply weights manually for XGBoost in .fit()
                sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
                model.fit(X_tr, y_train, sample_weight=sample_weights)
            else:
                model.fit(X_tr, y_train)
                
            y_pred = model.predict(X_ts)
            
            # ── Log Parameters ────────────────────────────────────────────────────────
            params = {
                "test_size"             : 0.2,
                "stratify"              : True,
                "random_state"          : 42,
                "representation"        : "TFIDF_500",
                "scaler"                : "MinMaxScaler" if is_nb else "StandardScaler",
                "model_name"            : model_name,
                "strategy_category"     : "Resampling" if resampler else strategy,
                "specific_strategy"     : strategy,
                "weights_applied"       : "balanced" if strategy == "ClassWeights" else "None",
            }
            
            if hasattr(curr_clf, 'get_params'):
                model_params = curr_clf.get_params()
                for k in ['C', 'alpha', 'n_estimators', 'max_depth', 'n_neighbors', 'kernel']:
                    if k in model_params:
                        params[f"model_{k}"] = str(model_params[k])
            
            mlflow.log_params(params)

            # ── Log Metrics ───────────────────────────────────────────────────────────
            report_dict:dict = classification_report(y_test, y_pred, output_dict=True) # type: ignore
            
            metrics = {
                "accuracy"         : accuracy_score(y_test, y_pred),
                "f1_weighted"      : report_dict['weighted avg']['f1-score'],
                "f1_macro"         : report_dict['macro avg']['f1-score'],
                "precision_weighted": report_dict['weighted avg']['precision'],
                "precision_macro"  : report_dict['macro avg']['precision'],
                "recall_weighted"  : report_dict['weighted avg']['recall'],
                "recall_macro"     : report_dict['macro avg']['recall'],
            }
            
            # Log Individual Class Metrics
            for i, label in enumerate(le.classes_):
                metrics[f"f1_class_{label}"] = report_dict[str(i)]['f1-score']
                metrics[f"precision_class_{label}"] = report_dict[str(i)]['precision']
                metrics[f"recall_class_{label}"] = report_dict[str(i)]['recall']
                
            mlflow.log_metrics(metrics)

            # ── Log Artifacts ─────────────────────────────────────────────────────────
            report_str = classification_report(y_test, y_pred, target_names=le.classes_.astype(str))
            print(f"Finished: {run_name} | F1-Macro: {metrics['f1_macro']:.4f}")

            with tempfile.TemporaryDirectory() as tmp_dir:
                report_path = os.path.join(tmp_dir, "classification_report.txt")
                with open(report_path, "w") as f:
                    f.write(str(report_str))
                mlflow.log_artifact(report_path)

                plt.figure(figsize=(8, 5))
                sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
                            xticklabels=le.classes_, yticklabels=le.classes_)
                plt.title(f'Confusion Matrix - {run_name}')
                plt.tight_layout()
                
                cm_path = os.path.join(tmp_dir, "confusion_matrix.png")
                plt.savefig(cm_path)
                mlflow.log_artifact(cm_path)
                plt.close()


Finished: LogisticRegression_Baseline | F1-Macro: 0.6640
🏃 View run LogisticRegression_Baseline at: http://localhost:5000/#/experiments/8/runs/438ede185607429eba81fc04106afa4e
🧪 View experiment at: http://localhost:5000/#/experiments/8
Finished: LogisticRegression_ClassWeights | F1-Macro: 0.6735
🏃 View run LogisticRegression_ClassWeights at: http://localhost:5000/#/experiments/8/runs/1efab81a23fa488f94e1085898d40144
🧪 View experiment at: http://localhost:5000/#/experiments/8
Finished: LogisticRegression_SMOTE | F1-Macro: 0.6643
🏃 View run LogisticRegression_SMOTE at: http://localhost:5000/#/experiments/8/runs/f270ff59e19940f3bfc01631c036db1a
🧪 View experiment at: http://localhost:5000/#/experiments/8
Finished: LogisticRegression_SMOTETomek | F1-Macro: 0.6627
🏃 View run LogisticRegression_SMOTETomek at: http://localhost:5000/#/experiments/8/runs/b6889bb5438344539c5b93157e2026f6
🧪 View experiment at: http://localhost:5000/#/experiments/8
Finished: NaiveBayes_Baseline | F1-Macro: 0.4742
🏃

c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Finished: LightGBM_Baseline | F1-Macro: 0.6516
🏃 View run LightGBM_Baseline at: http://localhost:5000/#/experiments/8/runs/f3998521b5a9418cac03331c7b25374d
🧪 View experiment at: http://localhost:5000/#/experiments/8


c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Finished: LightGBM_ClassWeights | F1-Macro: 0.6564
🏃 View run LightGBM_ClassWeights at: http://localhost:5000/#/experiments/8/runs/079b4e0a0ce54d2facb17c816199d285
🧪 View experiment at: http://localhost:5000/#/experiments/8


c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Finished: LightGBM_SMOTE | F1-Macro: 0.6672
🏃 View run LightGBM_SMOTE at: http://localhost:5000/#/experiments/8/runs/6f11a20b43f04cae81789affbe2d18a5
🧪 View experiment at: http://localhost:5000/#/experiments/8


c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Finished: LightGBM_SMOTETomek | F1-Macro: 0.6656
🏃 View run LightGBM_SMOTETomek at: http://localhost:5000/#/experiments/8/runs/0affb44219e44e38a9fd17c9ae9f1c5d
🧪 View experiment at: http://localhost:5000/#/experiments/8
Finished: XGBoost_Baseline | F1-Macro: 0.6277
🏃 View run XGBoost_Baseline at: http://localhost:5000/#/experiments/8/runs/46aa613cc86b47f68ded4955342b2444
🧪 View experiment at: http://localhost:5000/#/experiments/8
Finished: XGBoost_ClassWeights | F1-Macro: 0.6342
🏃 View run XGBoost_ClassWeights at: http://localhost:5000/#/experiments/8/runs/be60cdfc5530411a90aa165d5102fc65
🧪 View experiment at: http://localhost:5000/#/experiments/8
Finished: XGBoost_SMOTE | F1-Macro: 0.6321
🏃 View run XGBoost_SMOTE at: http://localhost:5000/#/experiments/8/runs/f28a9956b8c24c7aab4ef54d2ca21622
🧪 View experiment at: http://localhost:5000/#/experiments/8
Finished: XGBoost_SMOTETomek | F1-Macro: 0.6417
🏃 View run XGBoost_SMOTETomek at: http://localhost:5000/#/experiments/8/runs/2d9bde53de9

Conculusion - XGBOOST w class weight and Complement Navibayes - baseline are doing good.